In [62]:
# import pandas as pd
# from pathlib import Path
# from IPython.display import display
# import matplotlib.pyplot as plt

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)
# pd.set_option("display.max_colwidth", None)

# collision_path = Path("..") / "data" / "raw" / "dft-road-casualty-statistics-collision-last-5-years.csv"
# casualty_path = Path("..") / "data" / "raw" / "dft-road-casualty-statistics-casualty-last-5-years.csv"
# vehicle_path = Path("..") / "data" / "raw" / "dft-road-casualty-statistics-vehicle-last-5-years.csv"

# dfcol = pd.read_csv(
#     collision_path,
#     dtype={"collision_index": "string", "collision_ref_no": "string"},
# )
# dfcas = pd.read_csv(
#     casualty_path,
#     dtype={"collision_index": "string", "collision_ref_no": "string"},
# )
# dfveh = pd.read_csv(
#     vehicle_path,
#     dtype={
#         "collision_index": "string",
#         "collision_ref_no": "string",
#         "generic_make_model": "string",
#     },
# )

# tables = {"COLLISION": dfcol, "CASUALTY": dfcas, "VEHICLE": dfveh}

# print("============= INFO ==================")
# for name, df in tables.items():
#     print(f"\nТаблица: {name}")
#     print(f"Размер: {df.shape[0]} строк, {df.shape[1]} столбцов")
#     print(f"Память: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

#     print("\nТипы данных:")
#     display(df.dtypes.to_frame(name="Тип данных"))

#     print("Первые 5 строк:")
#     display(df.head(5))
#     print("Последние 5 строк:")
#     display(df.tail(5))

# print("\n============= MISSES ==================")
# for name, df in tables.items():
#     print(f"\nТаблица: {name}")

#     missing = pd.DataFrame({
#         "Количество пропусков": df.isna().sum(),
#         "Процент пропусков": (df.isna().mean() * 100).round(2),
#     }).sort_values(by="Процент пропусков", ascending=False)
#     display(missing)

#     total_missing = int(df.isna().sum().sum())
#     total_cells = int(df.size)
#     cols_with_missing = missing[missing["Количество пропусков"] > 0].index.tolist()
#     print(f"Всего пропусков: {total_missing}")
#     print(f"Общая доля пропусков: {total_missing / total_cells * 100:.4f}%")
#     print(f"Столбцы с пропусками: {cols_with_missing if cols_with_missing else 'нет'}")

# print("\nDUPLICATES")
# dup_rows = []
# for name, df in tables.items():
#     dups = int(df.duplicated().sum())
#     dup_rows.append({
#         "Таблица": name,
#         "Строк": len(df),
#         "Дубликатов": dups,
#         "Есть дубликаты": "Да" if dups > 0 else "Нет",
#     })
# display(pd.DataFrame(dup_rows).set_index("Таблица"))

# print("\nУНИКАЛЬНОСТЬ COLLISION_INDEX")
# display(pd.DataFrame([{
#     "Уникален": bool(dfcol["collision_index"].is_unique),
#     "Уникальных значений": int(dfcol["collision_index"].nunique()),
#     "Всего значений": int(dfcol["collision_index"].size),
# }], index=["COLLISION"]))

# print("\nПОВТОРЕНИЯ COLLISION_INDEX В VEHICLE / CASUALTY")
# rep_rows = []
# for name, df in [("VEHICLE", dfveh), ("CASUALTY", dfcas)]:
#     counts = df["collision_index"].value_counts()
#     rep_rows.append({
#         "Таблица": name,
#         "Уникальных ДТП": int(df["collision_index"].nunique()),
#         "ДТП с >1 записью": int((counts > 1).sum()),
#         "Макс. записей на ДТП": int(counts.max()),
#     })
# display(pd.DataFrame(rep_rows).set_index("Таблица"))

# print("\nСОСТАВНЫЕ ИДЕНТИФИКАТОРЫ")
# composite_keys = {
#     "VEHICLE": (dfveh, ["collision_index", "vehicle_reference"]),
#     "CASUALTY": (dfcas, ["collision_index", "casualty_reference"]),
# }
# ck_rows = []
# for name, (df, keys) in composite_keys.items():
#     if all(col in df.columns for col in keys):
#         dups = int(df.duplicated(subset=keys).sum())
#         ck_rows.append({
#             "Таблица": name,
#             "Ключ": " + ".join(keys),
#             "Дубликатов": dups,
#             "Ключ уникален": "Да" if dups == 0 else "Нет",
#         })
#     else:
#         ck_rows.append({
#             "Таблица": name,
#             "Ключ": " + ".join(keys),
#             "Дубликатов": None,
#             "Ключ уникален": "Нет нужных столбцов",
#         })
# display(pd.DataFrame(ck_rows).set_index("Таблица"))

# print("\nПРОВЕРКА СВЯЗЕЙ МЕЖДУ ТАБЛИЦАМИ")
# collision_ids = set(dfcol["collision_index"])
# rel_rows = []
# for name, df in [("VEHICLE", dfveh), ("CASUALTY", dfcas)]:
#     child_ids = set(df["collision_index"])
#     orphan_ids = child_ids - collision_ids
#     missing_ids = collision_ids - child_ids
#     rel_rows.append({
#         "Таблица": name,
#         "ДТП без записей": len(missing_ids),
#         "Записи без ДТП": len(orphan_ids),
#         "Всего ДТП": len(collision_ids),
#         "Всего записей": len(child_ids),
#     })
# display(pd.DataFrame(rel_rows).set_index("Таблица"))

# MISSING_RULES = {
#     # ---------- COLLISION ----------
#     "enhanced_collision_severity":                       {"na": [-1]},
#     "local_authority_district":                          {"na": [-1]},
#     "local_authority_ons_district":                      {"na": [-1]},
#     "local_authority_highway":                           {"na": [-1]},
#     "first_road_class":                                  {"na": [-1]},
#     "first_road_number":                                 {"na": [-1]},
#     "road_type":                                         {"na": [-1], "unknown": [9], "normal": [1, 2, 3, 6, 7, 12]},
#     "speed_limit":                                       {"na": [-1], "unknown": [99], "normal": [20, 30, 40, 50, 60, 70]},
#     "junction_detail_historic":                          {"na": [-1], "unknown": [99]},
#     "junction_detail":                                   {"na": [-1], "unknown": [99]},
#     "junction_control":                                  {"na": [-1], "unknown": [9]},
#     "second_road_class":                                 {"na": [-1], "unknown": [9]},
#     "second_road_number":                                {"na": [-1]},
#     "pedestrian_crossing_human_control_historic":        {"na": [-1], "unknown": [9]},
#     "pedestrian_crossing_physical_facilities_historic":  {"na": [-1], "unknown": [9]},
#     "pedestrian_crossing":                               {"na": [-1], "unknown": [99]},
#     "light_conditions":                                  {"na": [-1], "normal": [1, 4, 5, 6, 7]},
#     "weather_conditions":                                {"na": [-1], "unknown": [9], "normal": [1, 2, 3, 4, 5, 6, 7, 8]},
#     "road_surface_conditions":                           {"na": [-1], "unknown": [9], "normal": [1, 2, 3, 4, 5, 6, 7]},
#     "special_conditions_at_site":                        {"na": [-1], "unknown": [9]},
#     "carriageway_hazards_historic":                      {"na": [-1], "unknown": [9]},
#     "carriageway_hazards":                               {"na": [-1], "unknown": [99]},
#     "urban_or_rural_area":                               {"na": [-1], "normal": [1, 2, 3]},
#     "did_police_officer_attend_scene_of_collision":      {"na": [-1]},
#     "did_police_officer_attend_scene_of_accident":       {"na": [-1]},
#     "trunk_road_flag":                                   {"na": [-1]},
#     # ---------- VEHICLE ----------
#     "vehicle_type":                                      {"na": [-1], "unknown": [99]},
#     "towing_and_articulation":                           {"na": [-1], "unknown": [9]},
#     "vehicle_manoeuvre_historic":                        {"na": [-1], "unknown": [99]},
#     "vehicle_manoeuvre":                                 {"na": [-1], "unknown": [99]},
#     "vehicle_direction_from":                            {"na": [-1], "unknown": [9]},
#     "vehicle_direction_to":                              {"na": [-1], "unknown": [9]},
#     "vehicle_location_restricted_lane_historic":         {"na": [-1], "unknown": [99]},
#     "vehicle_location_restricted_lane":                  {"na": [-1], "unknown": [99]},
#     "junction_location":                                 {"na": [-1], "unknown": [9]},
#     "skidding_and_overturning":                          {"na": [-1], "unknown": [9]},
#     "hit_object_in_carriageway":                         {"na": [-1], "unknown": [99]},
#     "vehicle_leaving_carriageway":                       {"na": [-1], "unknown": [9]},
#     "hit_object_off_carriageway":                        {"na": [-1], "unknown": [99]},
#     "first_point_of_impact":                             {"na": [-1], "unknown": [9]},
#     "vehicle_left_hand_drive":                           {"na": [-1], "unknown": [9]},
#     "journey_purpose_of_driver_historic":                {"na": [-1], "unknown": [6]},
#     "journey_purpose_of_driver":                         {"na": [-1], "unknown": [6]},
#     "sex_of_driver":                                     {"na": [-1], "unknown": [3]},
#     "age_of_driver":                                     {"na": [-1]},
#     "age_band_of_driver":                                {"na": [-1]},
#     "engine_capacity_cc":                                {"na": [-1]},
#     "propulsion_code":                                   {"na": [-1]},
#     "generic_make_model":                                {"na": [-1]},
#     "driver_imd_decile":                                 {"na": [-1]},
#     "driver_distance_banding":                           {"na": [-1]},
#     # ---------- CASUALTY ----------
#     "sex_of_casualty":                                   {"na": [-1], "unknown": [9]},
#     "age_of_casualty":                                   {"na": [-1]},
#     "age_band_of_casualty":                              {"na": [-1]},
#     "enhanced_casualty_severity":                        {"na": [-1]},
#     "pedestrian_location":                               {"na": [-1], "unknown": [10]},
#     "pedestrian_movement":                               {"na": [-1], "unknown": [9]},
#     "car_passenger":                                     {"na": [-1], "unknown": [9]},
#     "bus_or_coach_passenger":                            {"na": [-1], "unknown": [9]},
#     "pedestrian_road_maintenance_worker":                {"na": [-1], "unknown": [2]},
#     "casualty_type":                                     {"na": [-1], "unknown": [99]},
#     "casualty_imd_decile":                               {"na": [-1]},
#     "casualty_distance_banding":                         {"na": [-1]},
# }


# def build_missing_report(tables: dict, rules: dict) -> pd.DataFrame:
#     rows = []
#     for table_name, df in tables.items():
#         n = len(df)
#         for col, spec in rules.items():
#             if col not in df.columns:
#                 continue

#             s = df[col]
#             s_num = pd.to_numeric(s, errors="coerce")

#             na_codes = spec.get("na", [])
#             unknown_codes = spec.get("unknown", [])

#             pct_nan = (s.isna().mean() * 100) if n else 0.0
#             pct_na = (s_num.isin(na_codes).mean() * 100) if (n and na_codes) else 0.0
#             pct_unknown = (s_num.isin(unknown_codes).mean() * 100) if (n and unknown_codes) else 0.0

#             rows.append({
#                 "Таблица": table_name,
#                 "Колонка": col,
#                 "% NaN": round(pct_nan, 2),
#                 "% NA (-1)": round(pct_na, 2),
#                 "% unknown (9/99)": round(pct_unknown, 2),
#             })

#     report = pd.DataFrame(rows)
#     if not report.empty:
#         report = report.sort_values(
#             by=["Таблица", "% NA (-1)", "% unknown (9/99)", "% NaN"],
#             ascending=[True, False, False, False],
#         ).reset_index(drop=True)
#     return report


# print("\n============= ПРОПУСКИ: NaN / NA-коды / unknown-коды =================")
# missing_report = build_missing_report(tables, MISSING_RULES)
# display(missing_report)


# def check_col(df: pd.DataFrame, col: str, rules: dict) -> tuple[str, list]:
#     if col not in df.columns:
#         return f"{col}: колонки нет в таблице — НЕ ОК", []
#     spec = rules.get(col)
#     if spec is None:
#         return f"{col}: нет правил в MISSING_RULES — НЕ ОК", []

#     allowed = set()
#     for key in ("na", "unknown", "normal"):
#         allowed |= set(spec.get(key, []))

#     s = pd.to_numeric(df[col], errors="coerce").dropna()
#     unexpected = sorted(set(s.unique()) - allowed)

#     if unexpected:
#         return f"{col}: НЕ ОК", unexpected
#     return f"{col}: всё ок", []


# print("=====Checking ranges and anomalous values=====")

# check_cols = [
#     "speed_limit",
#     "light_conditions",
#     "weather_conditions",
#     "road_type",
#     "road_surface_conditions",
#     "urban_or_rural_area",
# ]

# show_distribution = {
#     "speed_limit",
#     "light_conditions",
# }

# all_ok = True
# for col in check_cols:
#     status, bad = check_col(dfcol, col, MISSING_RULES)
#     print(f"\n{status}")
#     if bad:
#         all_ok = False
#         print(f"    недопустимые значения: {bad}")

#     if col in show_distribution:
#         vc = (
#             dfcol[col]
#             .value_counts(dropna=False)
#             .rename_axis("значение")
#             .reset_index(name="кол-во")
#         )
#         vc["доля, %"] = (vc["кол-во"] / len(dfcol) * 100).round(2)
#         print(f"    распределение {col} ({len(dfcol)} строк):")
#         display(vc)


# if all_ok:
#     print("ИТОГ: всё норм — все значения во всех проверенных колонках допустимы.")
# else:
#     print("ИТОГ: НЕ норм — есть недопустимые значения (см. выше).")

# print('====Diapazons====')
# cheks = ['number_of_vehicles', 'number_of_casualties', 'latitude',
#         'longitude', 'location_easting_osgr','location_northing_osgr']

# display(dfcol[cheks].describe().T)
# top10 = dfcol['number_of_casualties'].sort_values(ascending=False).head(10)
# print(top10)
# for idx, val in top10.items():
#     idx = dfcol.loc[idx, 'collision_index']
#     actual = (dfcas['collision_index'] == idx).sum()
#     status = "настоящее" if actual == val else "ОШИБКА ВВОДА"
#     print(f"collision_index={idx}: заявлено={val}, в dfcas={actual} -> {status}")

# dfcol['date_parsed'] = pd.to_datetime(
#     dfcol['date'],
#     format='%d/%m/%Y',
#     errors='coerce'
# )
# nat_mask = dfcol['date_parsed'].isna()
# n_nat = int(nat_mask.sum())
# print(f"NaT: {n_nat} из {len(dfcol)}")

# mismatch = dfcol["date_parsed"].dt.year != dfcol["collision_year"]
# if mismatch.any():
#     print('Error, года различаются')
#     print(dfcol.loc[mismatch, ["date_parsed", "collision_year"]])

# print(dfcol['time'].isna().sum())
# lengths = dfcol['time'].astype('string').str.len()
# print(lengths.value_counts(dropna=False).sort_index())

# dfcol['time_parsed'] = pd.to_datetime(
#     dfcol['time'],
#     format='%H:%M',
#     errors='coerce'
# )
# print(f"min hour: {dfcol['time_parsed'].dt.hour.min()}, max hour: {dfcol['time_parsed'].dt.hour.max()}")
# dtpbymonth = dfcol.groupby(dfcol['date_parsed'].dt.to_period("M")).size()
# display(dtpbymonth)

# per_m = dfcol.groupby(dfcol['date_parsed'].dt.to_period('M')).size()
# per_m = per_m.reindex(
#     pd.period_range(per_m.index.min(), per_m.index.max(), freq='M'),
#     fill_value=0,
# )

# fig, ax = plt.subplots(figsize=(12, 3))
# ax.plot(per_m.index.to_timestamp(), per_m.values, 'o')
# ax.grid(True)
# plt.show()
# tp = dfcol['time_parsed']
# hh = tp.dt.hour
# mm = tp.dt.minute
# is_h0 = hh.eq(0)
# n_h0        = int(is_h0.sum())
# n_0000      = int((is_h0 & mm.eq(0)).sum())
# n_0001_0059 = int((is_h0 & mm.between(1, 59)).sum())
# uniform_0000      = 1/60
# uniform_0001_0059 = 59/60
# print(f"Hour 0, всего строк     : {n_h0}")
# print(f"00:00            : {n_0000:6d}  ({n_0000/n_h0:6.2%}; при равномерном ожидалось {uniform_0000:.2%})")
# print(f"00:01..00:59     : {n_0001_0059:6d}  ({n_0001_0059/n_h0:6.2%}; при равномерном ожидалось {uniform_0001_0059:.2%})")

# subset = ["date_parsed", "time", "latitude", "longitude",
#         "number_of_vehicles","number_of_casualties", "police_force"]

# dups = dfcol[dfcol.duplicated(subset=subset, keep=False)]
# print(f"дублей: {len(dups)}\nБез координат:{dups["latitude"].isna().sum()}")
# df_geo = dfcol[dfcol["latitude"].notna() & dfcol["longitude"].notna()]
# print("после чистки:", df_geo.duplicated(subset=subset, keep=False).sum())

# veh = dfveh.groupby("collision_index").size()
# dfcol["veh_count"] = dfcol["collision_index"].map(veh)
# m_veh = dfcol["veh_count"].fillna(-1) != dfcol["number_of_vehicles"]

# cas = dfcas.groupby("collision_index").size()
# dfcol["cas_count"] = dfcol["collision_index"].map(cas)
# m_cas = dfcol["cas_count"].fillna(-1) != dfcol["number_of_casualties"]

# sev = dfcas.groupby("collision_index")["casualty_severity"].min()
# dfcol["sev_min"] = dfcol["collision_index"].map(sev)
# m_sev = dfcol["sev_min"].fillna(-1) != dfcol["collision_severity"]

# for name, m in [("veh", m_veh), ("cas", m_cas), ("sev", m_sev)]:
#     print(f"{name}: {m.sum()} ({m.mean():.3%}) из 513801 = {m.sum()/513801:.3%}")

# Выводы по аудиту collision

Код закоммичен, дабы не засорять пространство. Все выводы основаны на анализе файла при помощи pandas.

## 1. Состав и размер

Три таблицы: `collision` — 513 801, `casualty` — 937 265, `vehicle` — 652 821 строк, годы 2021–2025. `collision_index` уникален, у `vehicle` и `casualty` нет записей без ДТП и ДТП без записей. Состав и типы колонок сверены со словарём, расхождения: нет.

## 2. Настоящие пропуски (NaN)

| Колонки с NaN | Количество пропусков | Процент пропусков |
|---|---:|---:|
| `location_northing_osgr` | 53 | 0.01 |
| `latitude` | 53 | 0.01 |
| `location_easting_osgr` | 53 | 0.01 |
| `longitude` | 53 | 0.01 |

Заметных пропусков в остальных колонках нет. NaN оставляем как есть.

## 3. Коды -1 и unknown

`isna()` не видит -1, поэтому реальная доля пропусков выше.

**Больше всего -1:**

| № | Таблица | Колонка | Доля |
|---:|---|---|---:|
| 1 | COLLISION | `local_authority_district` | 99.96% |
| 2 | COLLISION | `junction_control` | 42.41% |
| 3 | COLLISION | `second_road_number` | 34.66% |

**Больше всего unknown (9/99):**

| № | Таблица | Колонка | Доля |
|---:|---|---|---:|
| 1 | COLLISION | `pedestrian_crossing_human_control_historic` | 3.53% |
| 2 | COLLISION | `pedestrian_crossing_physical_facilities_historic` | 2.97% |
| 3 | COLLISION | `weather_conditions` | 2.83% |

**Решение:** -1 → NaN только в колонках, где это задано в `MISSING_RULES`. Unknown остаётся отдельной категорией, потому что «неизвестно» тоже информация и для разных колонок оно кодируется по-разному.

## 4. Допустимые значения

Проверка по шести колонкам (`speed_limit`, `light_conditions`, `weather_conditions`, `road_type`, `road_surface_conditions`, `urban_or_rural_area`): всё ок. Доля -1 в `speed_limit`: 0.01%.
`speed_limit = 99` не существует в таблице.

## 5. Диапазоны

- **Координаты:** 0 строк вне диапазона Великобритании, широта и долгота не перепутаны. 53 строки без координат (во всех четырёх колонках сразу).
- `number_of_vehicles` (макс. 26) и `number_of_casualties` (макс. 142): нулей нет. Топ-10 по пострадавшим совпал с числом строк в `casualty`, значит, это реальные массовые ДТП, а не ошибки ввода.

**Решение:** строки без координат не удаляем, исключаем только из геоанализа. Выбросы оставляем. Из-за них среднее число пострадавших на ДТП искажено, поэтому используем медиану и доли.

## 6. Дата и время

`date` (DD/MM/YYYY) и `time` (HH:MM) распарсены без потерь: 0 NaT, 0 расхождений года с `collision_year`, пропусков во времени нет, часы 0–23. В часе 0 доля `00:00` равна 1.78% (при равномерном распределении ожидалось 1.67%), признаков заглушки нет. Данные есть за все 60 месяцев. В январе–феврале 2021 ДТП заметно меньше (около 5 500 против 9 000 и выше в других месяцах). Причина по данным не установлена; возможное объяснение — ограничения передвижения, нужно проверять по внешним источникам.

**Решение:** парсить с явным `format`, хранить `hour` как `int8`, исходные `date` и `time` в итоговом наборе не нужны.

## 7. Дубли по смыслу

По набору (дата, время, координаты, число ТС и пострадавших, `police_force`) найдено 6 строк в группах (3 лишние, ≈0,001%), все с координатами. Остальные поля в парах совпадают.

**Решение:** оставить; на статистику не влияет.

## 8. Согласованность таблиц

Число ТС, число пострадавших и тяжесть в `collision` полностью совпадают с агрегатами `vehicle` и `casualty` (0 расхождений из 513 801). `collision_severity` равна минимуму `casualty_severity` (1 = fatal).

**Решение:** агрегаты присоединяются к `collision` без дополнительных правил. Признаки из `casualty` определяют исход, поэтому в модель недели 7 их не берём (утечка данных).

## Список решений для `cleaning.py`

1. Заменить `-1` на `NaN` по словарю правил.
2. Распарсить `date` и `time` с явным форматом, получить год, месяц, день недели, час, выходной, время суток.
3. Привести типы (`category`, целые числа).
4. Расшифровать коды в названия, словари держать отдельно от логики.
5. Флаг наличия координат вместо удаления строк.
6. Агрегировать `vehicle` и `casualty` до уровня ДТП, `merge(how="left")`, проверить 513 801 строк и уникальность ключа.
7. Сохранить в Parquet.